# DIT 716 · Agentic Hydrological Diagnosis — Live Demo

**เป้าหมาย:** สาธิต agent 4 บทบาทตามงานออกแบบของเรา ตั้งแต่ตรวจคุณภาพข้อมูล → เพิ่มบริบทอุทกวิทยา → วินิจฉัยแยกสาเหตุ → ตรวจคำกล่าวอ้างกับ evidence ID โดย A4 และสาธิตการส่งความจำระยะยาวผ่าน memory gateway

| Step | สิ่งที่จะเห็น | คำถามหลัก |
|---|---|---|
| 1 | ข้อมูลสถานีสังเคราะห์ 4 เหตุการณ์ | Anomaly หมายถึงข้อมูลผิดเสมอหรือไม่? |
| 2 | **A1 Data Quality** คำนวณรูปแบบสัญญาณ | A1 บอกอะไรได้ และอะไรห้ามอนุมาน? |
| 3 | **A2 Hydrology Context** + retrieval จาก LTM | ความจำเก่าคือหลักฐานของเหตุการณ์ปัจจุบันหรือเป็นเพียง analogy? |
| 4 | **A3 Diagnostic** สร้างสาเหตุแข่งขันกัน | สมมติฐานแยกจากข้อเท็จจริงและผลยืนยันหรือไม่? |
| 5 | **A4 Verifier** ตรวจ evidence และ abstain | จะจัดการข้อกล่าวหาเรื่องการโจมตีที่ไม่มีหลักฐานอย่างไร? |
| 6 | Human review ก่อนเขียน LTM | ใครมีสิทธิ์อนุมัติและเพิกถอนความจำ? |
| 7 | เทียบ A3 fault-injection กับ P1 | Verifier ลดข้ออ้างที่ไม่มีหลักฐานด้วยต้นทุนอะไร? |

> ใช้ Python standard library และ LTM prototype ในโฟลเดอร์เดียวกัน ไม่เรียก LLM/API และไม่มีข้อมูลสถานีจริง ผลทุกอย่างเป็นการสาธิตเชิงกลไก ไม่ใช่ผลวิจัยเชิงประจักษ์


## Step 0 · ตั้งค่าและขอบเขตความปลอดภัย

Notebook นี้จำลอง agent แบบ deterministic เพื่อให้รันซ้ำได้โดยไม่ต้องมี API key. A1/A2/A3/A4 เป็น **role functions** ที่แยกสิทธิ์และผลลัพธ์กัน ไม่ใช่ LLM อิสระที่มี credentials.

Agent เห็นเฉพาะข้อมูลที่ tool ส่งคืน ณ `decision_day`; evaluator-only labels จะถูกสร้างในเซลล์สุดท้ายและไม่ถูกส่งให้ agent. เส้นทางทุกเส้นเป็น offline/read-only: ไม่มีการสั่งควบคุมสถานีหรือ OT.


In [1]:
from pathlib import Path
import sys, json, math, statistics

# หาโฟลเดอร์ที่มี ltm.py เพื่อให้รันได้ทั้งจาก package นี้และ parent folder
_candidates = [Path.cwd(), *Path.cwd().parents,
               Path.cwd() / "output" / "DIT716_LTM_Prototype",
               Path.cwd() / "DIT716_LTM_Prototype"]
ROOT = next((p for p in _candidates if (p / "ltm.py").is_file() and (p / "fixtures.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("เปิด notebook จากโฟลเดอร์ DIT716_LTM_Prototype หรือโฟลเดอร์โครงการ")
sys.path.insert(0, str(ROOT))
from ltm import MemoryGateway, RetrievalRequest

with (ROOT / "fixtures.json").open(encoding="utf-8") as f:
    MEMORY_CORPUS = json.load(f)
MEMORY = MemoryGateway(":memory:")
MEMORY.seed(MEMORY_CORPUS)
DECISION_DAY = MEMORY_CORPUS["decision_day"]
HUMAN_MODE = "auto"  # เปลี่ยนเป็น "interactive" เพื่อให้ notebook ถามก่อน publish ความจำ
print("พร้อม · memory episodes =", MEMORY.count("episodes"),
      "· vectors =", MEMORY.count("vectors"), "· decision day =", DECISION_DAY)


พร้อม · memory episodes = 15 · vectors = 13 · decision day = 365


## Step 1 · โลกจำลองและหลักฐานที่ agent ใช้ได้

สร้าง 4 เหตุการณ์ที่แยกกัน: น้ำขึ้นพร้อมฝนและสถานีข้างเคียง, ค่าค้าง, spike ที่ไม่มีบริบท, และ packet หาย/มาช้าโดยมี maintenance note ที่ยังไม่พร้อมใช้ ณ เวลาตัดสินใจ

อย่าปน **scenario ที่ agent สังเกตได้** กับป้ายกำกับเฉลย. ข้อมูลที่มาช้าจะส่งผ่าน context tool หลัง `available_day` เท่านั้น.


In [2]:
CASE_WORLD = {
 "C1": {"title":"Rapid rise with hydrological context", "station":"S17", "basin":"BASIN-1", "decision_day":365,
        "values_m":[1.20,1.27,1.51,1.91],
        "context":[
          {"evidence_id":"EV-C1-RAIN","available_day":365,"kind":"rainfall","statement":"Rainfall increased upstream during this event window.","tags":["rainfall_observed"]},
          {"evidence_id":"EV-C1-UP","available_day":365,"kind":"neighbor_gauge","statement":"An upstream gauge also rose within the locally plausible lag window.","tags":["upstream_rise","supports_genuine_event"]}]},
 "C2": {"title":"Repeated stage value", "station":"S17", "basin":"BASIN-1", "decision_day":365,
        "values_m":[1.46,1.46,1.46,1.46],
        "context":[
          {"evidence_id":"EV-C2-RAIN","available_day":365,"kind":"rainfall","statement":"No notable rainfall was recorded in this short window.","tags":["no_rainfall_corroboration"]},
          {"evidence_id":"EV-C2-NEIGHBOR","available_day":365,"kind":"neighbor_gauge","statement":"The neighboring gauge changed gradually rather than remaining flat.","tags":["neighbor_not_flat"]}]},
 "C3": {"title":"Abrupt spike with no corroborating context", "station":"S17", "basin":"BASIN-1", "decision_day":365,
        "values_m":[1.08,1.10,3.35,1.12], "context":[]},
 "C4": {"title":"Missing / delayed packets", "station":"S17", "basin":"BASIN-1", "decision_day":365,
        "values_m":[0.82,None,0.84,None],
        "context":[
          {"evidence_id":"EV-C4-PACKET","available_day":365,"kind":"telemetry","statement":"Receive timestamps show a delayed packet burst.","tags":["packet_delay_observed"]},
          {"evidence_id":"EV-C4-MAINT","available_day":366,"kind":"maintenance","statement":"A maintenance note records a modem reset after the event.","tags":["maintenance_reset"]}]}
}

EVIDENCE_LEDGER = {}
def read_available_context(case_id):
    """Tool boundary: release only records available by that case's decision time."""
    case = CASE_WORLD[case_id]
    available, withheld = [], []
    for row in case["context"]:
        (available if row["available_day"] <= case["decision_day"] else withheld).append(row)
    return available, withheld

print("กรณีจำลอง:")
for cid, case in CASE_WORLD.items():
    print(f"  {cid}: {case['title']} · readings={len(case['values_m'])} · context records={len(case['context'])}")


กรณีจำลอง:
  C1: Rapid rise with hydrological context · readings=4 · context records=2
  C2: Repeated stage value · readings=4 · context records=2
  C3: Abrupt spike with no corroborating context · readings=4 · context records=0
  C4: Missing / delayed packets · readings=4 · context records=2


## Step 2 · A1 Data Quality Agent — ตรวจ morphology ด้วยเครื่องมือเชิงตัวเลข

A1 คำนวณลักษณะของตัวอย่าง เช่น flatline, rapid rise, spike และ missingness แล้วบันทึกเป็น **observation evidence**. A1 ไม่ระบุสาเหตุ ไม่กล่าวหาเจตนาร้าย และไม่แก้ข้อมูลดิบ.


In [3]:
def a1_data_quality(case_id):
    case = CASE_WORLD[case_id]
    raw = case["values_m"]
    values = [x for x in raw if x is not None]
    deltas = [b-a for a,b in zip(values, values[1:])]
    flatline = len(values) >= 3 and len(set(values)) == 1
    rapid_rise = len(values) >= 3 and all(d >= 0 for d in deltas) and (values[-1]-values[0]) >= 0.5
    spike = bool(deltas) and max(abs(d) for d in deltas) >= 1.0
    missing = len(values) < len(raw)
    tags = []
    if flatline: tags.append("flatline_morphology")
    if rapid_rise: tags.append("rapid_rise_morphology")
    if spike: tags.append("spike_morphology")
    if missing: tags.append("missingness_morphology")
    if not tags: tags.append("no_simple_morphology_flag")
    eid = f"EV-{case_id}-A1"
    statement = (f"A1 observed {len(values)}/{len(raw)} numeric readings; "
                 f"flatline={flatline}, rapid_rise={rapid_rise}, spike={spike}, missing={missing}.")
    EVIDENCE_LEDGER[eid] = {"evidence_id":eid,"case_id":case_id,"producer":"A1",
                            "kind":"measurement_morphology","statement":statement,
                            "tags":tags,"available_day":case["decision_day"]}
    return {"case_id":case_id,"valid_count":len(values),"sample_count":len(raw),
            "flatline":flatline,"rapid_rise":rapid_rise,"spike":spike,"missing":missing,
            "max_step_m":round(max((abs(d) for d in deltas),default=0),3),"evidence_id":eid}

A1_RESULTS = {cid:a1_data_quality(cid) for cid in CASE_WORLD}
for cid, result in A1_RESULTS.items():
    print(cid, "→", {k:v for k,v in result.items() if k != "case_id"})
print("\nข้อจำกัด: morphology เป็นสิ่งที่วัดได้; สาเหตุยังเป็นสมมติฐาน")


C1 → {'valid_count': 4, 'sample_count': 4, 'flatline': False, 'rapid_rise': True, 'spike': False, 'missing': False, 'max_step_m': 0.4, 'evidence_id': 'EV-C1-A1'}
C2 → {'valid_count': 4, 'sample_count': 4, 'flatline': True, 'rapid_rise': False, 'spike': False, 'missing': False, 'max_step_m': 0.0, 'evidence_id': 'EV-C2-A1'}
C3 → {'valid_count': 4, 'sample_count': 4, 'flatline': False, 'rapid_rise': False, 'spike': True, 'missing': False, 'max_step_m': 2.25, 'evidence_id': 'EV-C3-A1'}
C4 → {'valid_count': 2, 'sample_count': 4, 'flatline': False, 'rapid_rise': False, 'spike': False, 'missing': True, 'max_step_m': 0.02, 'evidence_id': 'EV-C4-A1'}

ข้อจำกัด: morphology เป็นสิ่งที่วัดได้; สาเหตุยังเป็นสมมติฐาน


## Step 3 · A2 Hydrology Context Agent — บริบทปัจจุบันและความจำเป็น analogy

A2 ใช้ฝน สถานีข้างเคียง และ provenance ที่พร้อมใช้ ณ decision time. นอกจากนี้ A2 ค้น episodic memory ผ่าน gateway. ผล memory ถูกเก็บแยกเป็น `analogies` — ห้ามนำอดีตมาอ้างเป็นหลักฐานตรงของเหตุการณ์ปัจจุบัน.


In [4]:
MEMORY_QUERIES = {
 "C1":"S17 rapid stage rise after upstream rainfall neighboring gauge lag",
 "C2":"S17 stage gauge flatline frozen sensor repeated value",
 "C3":"S17 abrupt stage spike no corroborating hydrology context",
 "C4":"S17 maintenance modem reset delayed packet timestamp shift"
}

def a2_hydrology_context(case_id):
    case = CASE_WORLD[case_id]
    records, withheld = read_available_context(case_id)
    current_ids, current_tags = [], set()
    for row in records:
        eid = row["evidence_id"]
        EVIDENCE_LEDGER[eid] = {"evidence_id":eid,"case_id":case_id,"producer":"A2",
                                "kind":row["kind"],"statement":row["statement"],
                                "tags":row["tags"],"available_day":row["available_day"]}
        current_ids.append(eid); current_tags.update(row["tags"])
    request = RetrievalRequest(text=MEMORY_QUERIES[case_id], role="A2", basin=case["basin"],
        station=case["station"], query_day=case["decision_day"], observation_day=case["decision_day"],
        top_k=2, max_age_days=180)
    retrieved = MEMORY.retrieve(request)
    # The A2 agent receives only summaries returned by the policy gateway.
    analogies = [{"memory_id":x["memory_id"],"summary":x["summary"],"status":x["status"]}
                 for x in retrieved["items"]]
    missing = []
    if not any(EVIDENCE_LEDGER[e]["kind"] == "rainfall" for e in current_ids): missing.append("rainfall context")
    if not any(EVIDENCE_LEDGER[e]["kind"] == "neighbor_gauge" for e in current_ids): missing.append("neighbor-gauge context")
    if not any(EVIDENCE_LEDGER[e]["kind"] == "maintenance" for e in current_ids): missing.append("available maintenance record")
    return {"case_id":case_id,"current_evidence_ids":current_ids,"context_tags":sorted(current_tags),
            "missing_context":missing,"analogies":analogies,"withheld_count":len(withheld),
            "memory_exclusions":retrieved["excluded"]}

A2_RESULTS = {cid:a2_hydrology_context(cid) for cid in CASE_WORLD}
for cid, result in A2_RESULTS.items():
    print(f"{cid}: current IDs={result['current_evidence_ids']} · tags={result['context_tags']}")
    print("   memory analogies:", [x["memory_id"] for x in result["analogies"]],
          "· future records withheld:", result["withheld_count"],
          "· not-yet-available memory excluded:", result["memory_exclusions"]["not_yet_available"])


C1: current IDs=['EV-C1-RAIN', 'EV-C1-UP'] · tags=['rainfall_observed', 'supports_genuine_event', 'upstream_rise']
   memory analogies: ['EP-002', 'EP-008'] · future records withheld: 0 · not-yet-available memory excluded: 1
C2: current IDs=['EV-C2-RAIN', 'EV-C2-NEIGHBOR'] · tags=['neighbor_not_flat', 'no_rainfall_corroboration']
   memory analogies: ['EP-001', 'EP-006'] · future records withheld: 0 · not-yet-available memory excluded: 1
C3: current IDs=[] · tags=[]
   memory analogies: ['EP-008', 'EP-006'] · future records withheld: 0 · not-yet-available memory excluded: 1
C4: current IDs=['EV-C4-PACKET'] · tags=['packet_delay_observed']
   memory analogies: ['EP-015', 'EP-003'] · future records withheld: 1 · not-yet-available memory excluded: 1


## Step 4 · A3 Diagnostic Agent — differential diagnosis ไม่ใช่ผู้ตัดสินข้อเท็จจริง

A3 ผสาน observation จาก A1 กับ current evidence จาก A2 และคง alternative causes ไว้. เพื่อทดสอบ A4 เราจะใส่ **fault-injection 1 จุด** ให้ A3 อ้างว่า spike ยืนยันการโจมตี ทั้งที่มีเพียง morphology. นี่คือ mutant สำหรับทดสอบ ไม่ใช่พฤติกรรมที่เราต้องการจาก production agent.


In [5]:
def a3_diagnostic(case_id, a1, a2, unsafe_test_mutant=False):
    case = CASE_WORLD[case_id]
    evidence_ids = [a1["evidence_id"]] + list(a2["current_evidence_ids"])
    tags = set(a2["context_tags"])
    if a1["rapid_rise"] and "supports_genuine_event" in tags:
        cause, text, certainty, abstain = ("genuine hydrological event",
            "A rising stage is a candidate genuine event; rainfall and upstream-gauge context are consistent.",
            "candidate", False)
        alternatives = ["sensor/data-quality fault", "telemetry/configuration fault"]
    elif a1["flatline"]:
        cause, text, certainty, abstain = ("sensor/data-quality fault",
            "A flatline makes a sensor/data-quality fault a candidate; morphology alone does not confirm it.",
            "candidate", False)
        alternatives = ["stable local condition", "telemetry/replay issue"]
    elif a1["missing"] and "packet_delay_observed" in tags:
        cause, text, certainty, abstain = ("unknown / insufficient evidence",
            "Packet delay is observed, but the available evidence cannot distinguish telemetry from sensor causes.",
            "uncertain", True)
        alternatives = ["telemetry/configuration fault", "sensor fault"]
    elif a1["spike"] and unsafe_test_mutant:
        cause, text, certainty, abstain = ("suspicious manipulation",
            "Malicious manipulation confirmed from the spike morphology.", "confirmed", False)
        alternatives = []
    else:
        cause, text, certainty, abstain = ("unknown / insufficient evidence",
            "The available evidence does not support a cause-level conclusion.", "uncertain", True)
        alternatives = ["genuine event", "sensor/data-quality fault", "suspicious manipulation"]
    return {"case_id":case_id,"candidate_cause":cause,"claim":text,"certainty":certainty,
            "claim_kind":"hypothesis" if certainty != "confirmed" else "adjudicated_outcome",
            "evidence_ids":evidence_ids,"alternatives":alternatives,"abstain":abstain}

A3_DRAFTS = {cid:a3_diagnostic(cid,A1_RESULTS[cid],A2_RESULTS[cid],unsafe_test_mutant=(cid=="C3"))
             for cid in CASE_WORLD}
for cid, draft in A3_DRAFTS.items():
    print(f"{cid}: {draft['candidate_cause']} [{draft['certainty']}] · abstain={draft['abstain']}")
    print("   claim:", draft["claim"])
    print("   evidence IDs:", draft["evidence_ids"], "· alternatives:", draft["alternatives"])


C1: genuine hydrological event [candidate] · abstain=False
   claim: A rising stage is a candidate genuine event; rainfall and upstream-gauge context are consistent.
   evidence IDs: ['EV-C1-A1', 'EV-C1-RAIN', 'EV-C1-UP'] · alternatives: ['sensor/data-quality fault', 'telemetry/configuration fault']
C2: sensor/data-quality fault [candidate] · abstain=False
   claim: A flatline makes a sensor/data-quality fault a candidate; morphology alone does not confirm it.
   evidence IDs: ['EV-C2-A1', 'EV-C2-RAIN', 'EV-C2-NEIGHBOR'] · alternatives: ['stable local condition', 'telemetry/replay issue']
C3: suspicious manipulation [confirmed] · abstain=False
   claim: Malicious manipulation confirmed from the spike morphology.
   evidence IDs: ['EV-C3-A1'] · alternatives: []
C4: unknown / insufficient evidence [uncertain] · abstain=True
   claim: Packet delay is observed, but the available evidence cannot distinguish telemetry from sensor causes.
   evidence IDs: ['EV-C4-A1', 'EV-C4-PACKET'] · altern

## Step 5 · A4 Verifier — ตรวจ claim กับ evidence ledger แล้ว downgrade เมื่อจำเป็น

A4 ทำงานแยกจาก A3: ตรวจว่าทุก ID มีอยู่จริง อยู่ใน case เดียวกัน และพร้อมใช้ ณ decision time; ตรวจข้อจำกัดด้านการกล่าวหาเจตนาร้าย; ส่งกลับผล `traceable`, `downgraded`, หรือ `escalate`.

**A4 ไม่รับรองว่าสาเหตุจริงถูกต้อง** — ตรวจได้เพียงการรองรับด้วยหลักฐานที่อ้าง และส่งกรณีไม่แน่ใจให้ reviewer.


In [6]:
def a4_verify(draft):
    case_id = draft["case_id"]
    case = CASE_WORLD[case_id]
    refs = draft["evidence_ids"]
    missing_refs = [eid for eid in refs if eid not in EVIDENCE_LEDGER]
    wrong_case = [eid for eid in refs if eid in EVIDENCE_LEDGER and EVIDENCE_LEDGER[eid]["case_id"] != case_id]
    future_refs = [eid for eid in refs if eid in EVIDENCE_LEDGER and
                   EVIDENCE_LEDGER[eid]["available_day"] > case["decision_day"]]
    cited_tags = {tag for eid in refs if eid in EVIDENCE_LEDGER for tag in EVIDENCE_LEDGER[eid]["tags"]}
    violations = []
    if missing_refs: violations.append("evidence ID not found")
    if wrong_case: violations.append("evidence belongs to another case")
    if future_refs: violations.append("evidence unavailable at decision time")
    cause = draft["candidate_cause"]
    if cause == "suspicious manipulation" and "independent_malicious_provenance" not in cited_tags:
        violations.append("signal morphology cannot establish malicious intent")
    if draft["claim_kind"] == "adjudicated_outcome":
        violations.append("agent cannot publish an adjudicated outcome")
    blocked = bool(violations)
    abstain = draft["abstain"] or blocked
    final_candidate = "unknown / insufficient evidence" if blocked else cause
    needs_review = abstain or blocked
    return {"case_id":case_id,"verdict":"downgraded / escalate" if blocked else ("escalate" if abstain else "traceable hypothesis"),
            "candidate_after_verification":final_candidate,"abstain":abstain,"needs_human_review":needs_review,
            "violations":violations,"cited_ids_valid":not (missing_refs or wrong_case or future_refs),
            "verified_true_cause":False}

A4_RESULTS = {cid:a4_verify(draft) for cid,draft in A3_DRAFTS.items()}
for cid, result in A4_RESULTS.items():
    print(f"{cid}: {result['verdict']} → {result['candidate_after_verification']} · abstain={result['abstain']}")
    if result["violations"]: print("   flags:", "; ".join(result["violations"]))
    print("   citations valid:", result["cited_ids_valid"], "· human review:", result["needs_human_review"],
          "· true cause certified:", result["verified_true_cause"])


C1: traceable hypothesis → genuine hydrological event · abstain=False
   citations valid: True · human review: False · true cause certified: False
C2: traceable hypothesis → sensor/data-quality fault · abstain=False
   citations valid: True · human review: False · true cause certified: False
C3: downgraded / escalate → unknown / insufficient evidence · abstain=True
   flags: signal morphology cannot establish malicious intent; agent cannot publish an adjudicated outcome
   citations valid: True · human review: True · true cause certified: False
C4: escalate → unknown / insufficient evidence · abstain=True
   citations valid: True · human review: True · true cause certified: False


### ดูเคสสำคัญ: morphology เหมือนกัน ไม่ได้พิสูจน์การโจมตี

C3 มี spike จริงในข้อมูลจำลอง แต่ไม่มี independent provenance. A3 test mutant เขียนว่า “ยืนยัน manipulation”; A4 ตรวจแล้ว downgrade เป็น `Unknown / insufficient evidence` และส่ง reviewer. นี่เป็นการทดสอบ policy ด้วยข้อผิดพลาดที่ตั้งใจสร้าง ไม่ใช่การวัดว่า A4 แก้ hallucination ของ LLM จริงได้กี่เปอร์เซ็นต์.


## Step 6 · Human review ก่อนเขียนสรุปลง episodic memory

A3/A4 เพิ่มได้เฉพาะ proposed episode. การอนุมัติและ vectorization ต้องผ่าน reviewer role; agent A3 ไม่มีสิทธิ์ยืนยันผลของตัวเอง. เซลล์นี้ทดลองกับ C1 ซึ่ง A4 ตรวจว่ามี current evidence รองรับเป็น **candidate** แต่ยังไม่เรียกว่าความจริงที่ยืนยันแล้ว.


In [7]:
# ผูกหลักฐานสังเคราะห์จาก C1 ให้ gateway ตรวจ lineage ได้
with MEMORY.conn:
    MEMORY.conn.execute("INSERT OR IGNORE INTO sources(source_id,kind,status) VALUES(?,?,?)",
                         ("SRC-M3-C1","synthetic_demo_case","active"))
    for evidence_id in A3_DRAFTS["C1"]["evidence_ids"]:
        MEMORY.conn.execute("INSERT OR IGNORE INTO evidence_records(evidence_id,source_id) VALUES(?,?)",
                             (evidence_id,"SRC-M3-C1"))

proposed_id = "EP-M3-DEMO-C1"
proposed_text = MEMORY.propose_summary(memory_id=proposed_id, station="S17", basin="BASIN-1",
    observed_morphology="gradual rapid stage rise after rainfall",
    candidate_hypothesis="genuine hydrological event is plausible; not an adjudicated outcome",
    source_ids=["SRC-M3-C1"], evidence_ids=A3_DRAFTS["C1"]["evidence_ids"], producer_role="A3",
    split="train", snapshot_id=MEMORY_CORPUS["snapshot_id"], day=DECISION_DAY)
print("proposal status:", MEMORY.conn.execute("SELECT status FROM episodes WHERE memory_id=?",(proposed_id,)).fetchone()[0])
print("vector exists before human approval:", bool(MEMORY.conn.execute("SELECT 1 FROM vectors WHERE memory_id=?",(proposed_id,)).fetchone()))
try:
    MEMORY.approve_episode(proposed_id, reviewer_id="A3", reviewer_role="A3", day=DECISION_DAY)
except PermissionError as exc:
    print("A3 approval denied:", exc)

if HUMAN_MODE == "interactive":
    human_decision = input("Reviewer: approve this candidate summary? [y/N] ").strip().lower()
    do_approve = human_decision == "y"
else:
    do_approve = True  # simulated reviewer for a repeatable classroom walkthrough
    print("Simulated reviewer: approves source-linked candidate summary for the demo.")
if do_approve:
    MEMORY.approve_episode(proposed_id, reviewer_id="reviewer-demo", reviewer_role="reviewer", day=DECISION_DAY)
    exact = MEMORY.retrieve_by_id(proposed_id, RetrievalRequest("candidate C1", role="A3", basin="BASIN-1",
        station="S17", query_day=DECISION_DAY, observation_day=DECISION_DAY,
        snapshot_id=MEMORY_CORPUS["snapshot_id"]))
    print("published after review:", exact["decision"], "· status:", exact["item"]["status"] if exact["item"] else None)
else:
    print("Not approved: proposal stays pending and is not vectorized.")


proposal status: pending
vector exists before human approval: False
A3 approval denied: Only a reviewer/steward can approve an episode
Simulated reviewer: approves source-linked candidate summary for the demo.
published after review: allow · status: approved


## Step 7 · Evaluator-only comparison — A3 fault mutant ก่อน/หลัง A4

เซลล์นี้เป็นส่วนประเมินผลเท่านั้น: เฉลย C1–C4 ไม่ถูกส่งเข้า A1/A2/A3/A4. ค่าที่แสดงนับจาก fixture 4 กรณีเพื่อใช้สอนการอ่าน audit trail ไม่ใช่ค่าประมาณประสิทธิภาพทั่วไป.


In [8]:
# Evaluator-only labels: intentionally not passed to any agent or memory query.
EVALUATOR_ONLY_LABELS = {
 "C1":{"expected":"genuine hydrological event candidate","malicious_provenance":False},
 "C2":{"expected":"sensor/data-quality fault candidate","malicious_provenance":False},
 "C3":{"expected":"unknown / insufficient evidence","malicious_provenance":False},
 "C4":{"expected":"unknown at decision time","malicious_provenance":False}
}

def compare_unverified_and_p1():
    raw_false_intent = 0
    blocked_false_intent = 0
    for cid, draft in A3_DRAFTS.items():
        truth = EVALUATOR_ONLY_LABELS[cid]
        if draft["candidate_cause"] == "suspicious manipulation" and not truth["malicious_provenance"]:
            raw_false_intent += 1
        final_cause = A4_RESULTS[cid]["candidate_after_verification"]
        if final_cause == "suspicious manipulation" and not truth["malicious_provenance"]:
            blocked_false_intent += 1
    return [
      {"condition":"A3 test mutant · no A4","unsupported intent claims":raw_false_intent,
       "abstentions":sum(d["abstain"] for d in A3_DRAFTS.values()),"review escalations":0},
      {"condition":"P1 · A4 + escalation","unsupported intent claims":blocked_false_intent,
       "abstentions":sum(r["abstain"] for r in A4_RESULTS.values()),
       "review escalations":sum(r["needs_human_review"] for r in A4_RESULTS.values())}
    ]

for row in compare_unverified_and_p1():
    print(f"{row['condition']:30} · unsupported intent={row['unsupported intent claims']}"
          f" · abstain={row['abstentions']} · human review={row['review escalations']}")
print("\nAudit note: A4 checked citations and rules; it did not certify any true hydrological cause.")


A3 test mutant · no A4         · unsupported intent=1 · abstain=1 · human review=0
P1 · A4 + escalation           · unsupported intent=0 · abstain=2 · human review=2

Audit note: A4 checked citations and rules; it did not certify any true hydrological cause.


## สรุปสำหรับการ demo และคำถามอภิปราย

- **บทบาทไม่ซ้ำกัน:** A1 วัดสัญญาณ, A2 เติมบริบท, A3 เสนอ differential diagnosis, A4 ตรวจ traceability และจำกัดข้อสรุป. Memory gateway เป็น service ไม่ใช่ agent ตัวที่ห้า.
- **หลักฐานคนละชนิด:** measurement morphology และ retrieved episode เป็นคนละรายการ; memory analogy ไม่ใช่หลักฐานตรงของเหตุการณ์ปัจจุบัน.
- **การควบคุม:** ไม่มี agent ใดออกคำสั่งควบคุม. การเผยแพร่ long-term episode ต้องให้ reviewer อนุมัติ; source revocation เป็นสิทธิ์ steward.
- **ผลใน toy fixture:** A4 ปิดข้อกล่าวหา manipulation ที่ไม่มี provenance แต่เพิ่มการ abstain/escalate. นี่เป็นผลที่ออกแบบให้เห็นในตัวอย่างจิ๋ว ไม่ใช่หลักฐานเชิงสถิติ.
- **ข้อจำกัด:** agents เป็นกฎจำลอง, 4 cases สังเคราะห์, LTM ใช้ deterministic hash vectors, ไม่มี real LLM, ผู้ใช้จริง หรือ data-owner review.

### คำถามให้นักศึกษาลองตอบ

1. A4 ควรตรวจอะไรเอง และอะไรควรส่งให้ domain reviewer?
2. ถ้าข้อมูลฝนขาดแต่ stage เพิ่มขึ้น A3 ควรสรุปหรือ abstain อย่างไร?
3. จะป้องกัน training-memory analogy ชี้นำการวินิจฉัยเหตุการณ์ใหม่เกินไปได้อย่างไร?
4. metric ใดควรเพิ่มก่อนใช้ข้อมูลจริง: evidence faithfulness, macro-F1, calibration, review time หรือ cost?

**รันซ้ำ:** `Kernel → Restart & Run All` · ใช้ Python 3.10+ และ package `DIT716_LTM_Prototype` ที่มี `ltm.py` กับ `fixtures.json`.


In [9]:
# พื้นที่ทดลอง: ลองปิดตัวทดสอบ A3 ที่ตั้งใจทำผิด แล้วรัน A3/A4 อีกครั้ง
SAFE_DRAFTS = {cid:a3_diagnostic(cid,A1_RESULTS[cid],A2_RESULTS[cid],unsafe_test_mutant=False)
               for cid in CASE_WORLD}
print("C3 without injected overclaim:", SAFE_DRAFTS["C3"]["candidate_cause"],
      "· abstain =", SAFE_DRAFTS["C3"]["abstain"])
print("C3 with test mutant + A4:", A4_RESULTS["C3"]["candidate_after_verification"],
      "· escalated =", A4_RESULTS["C3"]["needs_human_review"])


C3 without injected overclaim: unknown / insufficient evidence · abstain = True
C3 with test mutant + A4: unknown / insufficient evidence · escalated = True
